## 课程三：韵律控制（45%）

实验目的：了解 VITS 模型的结构和工作原理，学习 MeloTTS 的韵律控制机制

MeloTTS 的模型架构和实现参考了 [VITS](https://github.com/jaywalnut310/vits), [VITS2](https://github.com/daniilrobnikov/vits2) and [Bert-VITS2](https://github.com/fishaudio/Bert-VITS2) 等开源项目。在本次课程中，你需要了解 VITS 模型的结构和工作原理，并尝试控制 MeloTTS 的韵律。

请阅读和观看以下资料：
- [VITS 论文](https://arxiv.org/abs/2106.06103)
- [VITS 讲解视频](https://www.bilibili.com/video/BV1Jb4y1g7q6/?p=3&share_source=copy_web&vd_source=d0dab92918e0499fec84fd963e2a879e)

![vits](./vits.png)

> 【问题一】 请介绍一下 VITS 模型的组成结构，训练和推理过程的工作原理。

VITS，全称是 Variational Inference with adversarial learning for end-to-end Text-to-Speech，是一种端到端并行文本转语音模型。它的核心思想是把传统 TTS 系统中“文本到声学特征”和“声学特征到波形”这两个阶段合并到一个统一模型中训练，直接从文本生成原始语音波形。论文中将 VITS 表述为一个条件变分自编码器，也就是 conditional VAE，同时引入 normalizing flow 来增强先验分布的表达能力，并使用对抗训练来提升波形生成质量。 
VITS 的整体结构主要由后验编码器、先验编码器、解码器、判别器和随机时长预测器组成。论文第 3 页的 Figure 1 给出了训练和推理流程图，可以看到训练阶段使用后验编码器、先验编码器、normalizing flow、解码器、判别器和随机时长预测器；而推理阶段不再使用后验编码器和判别器，只保留文本编码、时长预测、flow 逆变换和解码器生成波形的路径。 
后验编码器的输入是目标语音的线性频谱，它的作用是在训练阶段从真实语音中提取潜变量 z。VITS 使用线性频谱而不是 mel 频谱作为后验编码器输入，是因为线性频谱包含更高分辨率的信息，有助于提升语音合成质量。后验编码器输出一个正态分布的均值和方差，然后从这个分布中采样潜变量 z。这个潜变量可以理解为真实语音在隐空间中的表示，它连接了文本端和波形端。 
先验编码器的输入是文本经过预处理后的音素序列。文本首先被转换为 IPA 音素序列，并插入 blank token。先验编码器中的文本编码器采用 Transformer encoder，并使用相对位置表示。文本编码器输出隐藏表示后，再通过线性投影得到先验分布的均值和方差。为了让先验分布更灵活，VITS 在先验编码器中加入 normalizing flow。normalizing flow 可以把一个简单的正态分布通过可逆变换变成更复杂的分布，从而更好地拟合真实语音潜变量的分布。论文的消融实验显示，去掉 normalizing flow 会导致 MOS 分数明显下降，说明它对合成质量非常关键。 
解码器的作用是把潜变量 z 上采样并转换成原始语音波形。VITS 的解码器基本采用 HiFi-GAN V1 generator 的结构，由多层转置卷积和多感受野融合模块组成。它不是先生成 mel 频谱再交给声码器，而是直接从潜变量生成 raw waveform。因此，VITS 可以避免传统两阶段 TTS 中声学模型和声码器分开训练、误差累积以及部署复杂的问题。 
判别器只在训练阶段使用。VITS 采用了 HiFi-GAN 中的 multi-period discriminator 思路，让判别器从不同周期模式上判断生成波形和真实波形的差异。训练时，判别器学习区分真实语音和生成语音，解码器则学习生成更像真实语音的波形。除了普通的对抗损失，VITS 还使用 feature matching loss，也就是让生成语音和真实语音在判别器中间层的特征尽量接近。这样可以提高训练稳定性，并改善语音自然度。 
随机时长预测器是 VITS 的一个重要创新。文本到语音不是一一对应的，同一句话可以用不同语速、节奏和停顿方式说出来。传统非自回归 TTS 常使用确定性时长预测器，对同一文本通常只能预测固定时长，难以产生自然的节奏变化。VITS 使用 flow-based stochastic duration predictor，从文本条件中采样音素时长，使模型能够生成具有不同节奏的语音。论文第 7 页的图表显示，Glow-TTS 由于使用确定性时长预测器，只能生成固定长度的语音，而 VITS 能产生更接近真实分布的语音长度变化。 
在训练过程中，VITS 首先把真实语音转换为线性频谱，并输入后验编码器，得到潜变量 z 的后验分布。与此同时，输入文本被转换为音素序列，经过文本编码器得到文本表示。由于训练数据中没有显式的音素到语音帧的对齐标注，VITS 使用 Monotonic Alignment Search，也就是 MAS，来估计文本音素和潜变量序列之间的单调对齐关系。这个对齐矩阵表示每个音素对应多少个语音时间步。MAS 的搜索目标是在单调且不跳过文本的约束下，找到能最大化潜变量先验似然的对齐路径。 
得到对齐关系后，模型可以计算每个音素的时长，并用这些时长训练随机时长预测器。由于音素时长本身是离散整数，而 normalizing flow 更适合处理连续变量，VITS 使用变分去量化和变分数据增强，把离散时长转换为适合 flow 建模的连续形式。训练时，随机时长预测器学习给定文本条件下的时长分布，而不是只学习一个固定值。 
训练目标由多个部分组成。重建损失用于约束生成语音转换得到的 mel 频谱接近真实语音的 mel 频谱。KL 损失用于让后验编码器得到的潜变量分布接近文本条件下的先验分布。时长损失用于训练随机时长预测器。对抗损失用于让生成波形更难被判别器区分。特征匹配损失用于让生成波形和真实波形在判别器中间层特征上保持一致。论文把这些损失合并为总的 VAE-GAN 训练目标。 
在推理过程中，VITS 不需要真实语音输入，因此后验编码器不会被使用。模型首先把输入文本转换成音素序列，并送入文本编码器得到文本隐藏表示。然后随机时长预测器根据文本表示和随机噪声采样每个音素的持续时间，再根据这些时长把文本表示扩展到语音时间轴上。接着，模型从文本条件下的先验分布中采样潜变量，并通过 normalizing flow 的逆变换得到适合解码器使用的潜变量 z。最后，解码器把 z 直接转换成原始语音波形。整个推理过程是并行的，不像自回归模型那样逐帧或逐采样点生成，因此速度很快。 
从工作原理上看，VITS 把 TTS 问题建模为“给定文本条件，生成语音潜变量，再由潜变量生成波形”的条件生成问题。训练时，模型通过真实语音学习潜变量空间、文本对齐、音素时长分布和波形生成能力；推理时，模型只依赖文本和随机噪声生成潜变量与时长，从而直接合成语音。由于引入了 VAE，模型可以表达文本到语音的一对多关系；由于引入了 normalizing flow，先验分布有更强的表达能力；由于引入了 GAN 训练，生成波形的自然度和细节更好；由于引入了随机时长预测器，语音节奏更加多样和自然。 
总体来说，VITS 的结构可以理解为一个端到端的 VAE-GAN TTS 系统。后验编码器在训练时从真实语音提取潜变量，先验编码器从文本预测潜变量分布，MAS 负责自动学习文本和语音之间的单调对齐，随机时长预测器负责建模音素持续时间的不确定性，normalizing flow 负责增强先验分布表达能力，HiFi-GAN 风格的解码器负责从潜变量直接生成波形，判别器负责通过对抗学习提升语音质量。它的主要优势是省去了传统两阶段 TTS 中显式的中间声学特征生成流程，实现了从文本到波形的统一训练和并行推理。 


> 【问题二】 请介绍一下 VITS 中的 Monotonic Alignment Search (MAS) 算法和 Stochastic Duration Predictor 模块，说明它们在时长建模和韵律生成中的作用。



在 VITS 中，Monotonic Alignment Search，也就是 MAS，和 Stochastic Duration Predictor，也就是随机时长预测器，是连接文本序列与语音时间序列的两个关键部分。它们共同解决了文本到语音中的一个核心问题：输入文本的长度和输出语音的帧数并不相等，模型必须知道每个音素应该对应多少个语音时间步，并且还要让同一句话能够以不同语速、停顿和节奏被自然地说出来。MAS 主要负责在训练阶段自动寻找文本音素和语音潜变量之间的对齐关系，而随机时长预测器主要负责学习并生成音素时长分布，从而在推理阶段产生自然且多样的韵律。

MAS 的作用可以理解为“自动对齐”。在 TTS 任务中，输入是音素序列，输出是语音序列，但训练数据通常只有整句话的文本和对应音频，并没有标注每个音素从哪一帧开始、到哪一帧结束。因此，VITS 需要在训练过程中自己估计音素与语音帧之间的对应关系。MAS 假设人类读文本时通常是按顺序发音的，不会倒着读，也不会跳过音素，所以它把对齐路径限制为单调且不跳跃的路径。也就是说，前面的音素只能对应前面的语音片段，后面的音素只能对应后面的语音片段，每个音素至少要按顺序被分配到相应的时间区域。

在 VITS 的训练过程中，后验编码器先从真实语音的线性频谱中得到潜变量 z，文本编码器则从音素序列中得到文本表示，并通过先验编码器预测每个文本位置对应的先验分布。MAS 要做的事情，就是在所有满足单调约束的候选对齐中，寻找一条最优路径，使得潜变量 z 在文本条件先验分布下的似然最大。论文中指出，虽然 VITS 的整体目标是最大化 ELBO，而不是直接最大化数据似然，但在推导后，寻找最优对齐的问题可以简化为寻找使潜变量先验概率最大的对齐路径，因此可以沿用 Glow-TTS 中的 MAS 动态规划实现。

从算法过程看，MAS 会构造一个文本长度乘以潜变量长度的得分矩阵。矩阵中的每个位置表示某个语音潜变量时间步对齐到某个音素时的对数似然得分。然后算法使用动态规划，在只能向前推进、不能回退、不能跳过文本顺序的条件下，累计每条可能路径的得分，并保留得分最高的路径。最后通过回溯得到一个 hard monotonic attention matrix，也就是硬单调对齐矩阵。这个矩阵中，每一列对应一个语音时间步，每一行对应一个音素，值为 1 的位置表示该时间步被分配给对应音素。论文附录第 12 页给出了 MAS 的伪代码，展示了它如何先计算最优累计得分，再从末尾回溯得到最终对齐路径。

MAS 得到的对齐矩阵不仅用于训练先验分布，还可以直接转换为音素时长。具体来说，一个音素对应的时长等于它在对齐矩阵中被分配到的语音时间步数量。换句话说，如果某个音素在对齐矩阵的一行中覆盖了 5 个时间步，那么这个音素的时长就是 5。这样，VITS 就在没有人工时长标注的情况下，从文本和语音配对数据中自动提取出了音素级时长信息。这个时长信息随后会被用来训练随机时长预测器。

Stochastic Duration Predictor 的作用可以理解为“学习文本对应的时长分布”，而不是只预测一个固定时长。传统非自回归 TTS 模型通常使用 deterministic duration predictor，也就是确定性时长预测器。确定性预测器对同一个文本输入往往只能输出同一组音素时长，因此同一句话每次合成出来的节奏基本一致。这种方式虽然稳定，但很难表现真实人类说话中的自然变化。实际上，同一句话可以说得快一点，也可以说得慢一点；某些音节可以拉长，某些停顿可以更明显。这种一对多关系正是语音韵律自然性的来源之一。VITS 因此设计了基于 flow 的随机时长预测器，让模型能够从一个条件分布中采样时长，从而产生多样化的语音节奏。

随机时长预测器的训练目标来自 MAS 估计出的音素时长。模型给定文本编码器输出的隐藏表示 htext，学习每个音素时长 d 的概率分布。由于音素时长是离散整数，而 normalizing flow 通常建模连续变量，所以 VITS 使用了变分去量化方法。它引入随机变量 u，把离散时长 d 转换成连续的 d 减 u，使其可以被连续流模型处理。同时，VITS 还引入另一个随机变量 ν，用于变分数据增强，把原本一维的时长标量扩展到更高维的表示空间。这样做可以缓解标量时长难以进行高维可逆变换的问题，使 flow 模型有更强的表达能力。

在训练随机时长预测器时，模型会用一个近似后验分布来采样 u 和 ν，并通过 flow 计算增强和去量化后的时长数据在文本条件下的似然。训练损失是时长对数似然下界的负值。论文还特别提到，在训练时会对输入条件使用 stop gradient，也就是停止梯度传播，使随机时长预测器的训练不会反向影响文本编码器等其他模块的学习。这样可以让时长模块独立学习时长分布，同时避免破坏主干生成模型的表示学习。

在推理阶段，MAS 不再使用，因为推理时没有真实语音，也就无法从真实语音中估计对齐。此时，随机时长预测器接替了 MAS 在时长生成上的作用。模型先将输入文本转换为音素序列，并通过文本编码器得到 htext。然后随机时长预测器从随机噪声出发，通过 flow 的逆变换生成每个音素的持续时间，再把连续结果转换为整数时长。根据这些预测出的音素时长，模型就可以把文本表示扩展到与语音时间轴一致的长度，进一步生成潜变量并由解码器合成原始波形。论文第 3 页的 Figure 1 展示了这一点：训练时 MAS 从潜变量和文本先验之间寻找对齐，并将得到的时长用于训练随机时长预测器；推理时随机时长预测器直接根据文本和噪声生成时长，之后再进入 flow 逆变换和解码器生成波形。

MAS 和随机时长预测器在时长建模中是前后衔接的关系。MAS 负责从训练数据中“发现”音素时长，它相当于一个无监督对齐器，为每个音素提供训练目标。随机时长预测器负责从这些时长样本中“学习”条件分布，它不是简单记住一个平均时长，而是学习在给定文本条件下可能出现的多种时长变化。这样，VITS 既不需要外部强制对齐工具，也不需要人工标注音素边界，就能完成音素级时长建模。

在韵律生成方面，随机时长预测器的意义尤其重要。语音韵律不仅包括音高变化，也包括语速、音节拉伸、停顿和节奏。时长预测直接决定了每个音素在时间轴上占据多长，因此会显著影响语音节奏。如果时长预测过于固定，生成语音会显得机械；如果时长具有合理的随机性，模型就可以对同一句文本生成不同但自然的表达方式。论文第 7 页的 Figure 2 显示，Glow-TTS 使用确定性时长预测器时，对同一句话只能产生一个固定长度，而 VITS 可以生成具有分布变化的语音长度，并且在多说话人场景下还能体现不同说话人的时长习惯。

从整体上看，MAS 保证了训练阶段文本和语音之间的可靠单调对齐，使模型知道“哪个音素对应哪段语音”；随机时长预测器则把 MAS 得到的时长进一步建模成可采样的概率分布，使推理阶段可以生成自然、多样的音素持续时间。MAS 解决的是训练时的对齐学习问题，Stochastic Duration Predictor 解决的是推理时的时长生成和节奏多样性问题。二者结合后，VITS 能够在非自回归、并行生成的框架下，同时保持稳定对齐和自然韵律，这是它相比许多传统非自回归 TTS 模型的重要优势。


MeloTTS 使用 Stochastic Duration Predictor 模块来实现韵律控制机制的代码在 `melo/models.py` 的 `infer` 函数中（966 行），其中一个重要的输出是 `w_ceil`，它表示了每个音素的持续时间。可以通过自定义的 `get_original_w_ceil` 函数来获取 `w_ceil`：

In [1]:
from melo.api import TTS
import pandas as pd

pd.set_option('display.max_columns', None)

# Speed is adjustable
speed = 1
device = 'cpu' # or cuda:0

text = "落霞与孤鹜齐飞，秋水共长天一色。"
model = TTS(language='ZH', device=device)
speaker_ids = model.hps.data.spk2id

output_path = 'zh.wav'

# 获取原始的 phones、tones、w_ceil 列表
w_ceil_list, phone_list, tone_list = model.get_original_w_ceil(text, speaker_ids['ZH'], output_path, speed=speed, sdp_ratio=0, noise_scale=0, noise_scale_w=0)

# 将 phones 列表中的 ID 转换为对应的符号
symbol_to_id_map = model.symbol_to_id
id_to_symbol_map = {v: k for k, v in zip(symbol_to_id_map.keys(), symbol_to_id_map.values())}
print(f'id_to_symbol_map: {id_to_symbol_map}')
# for w_ceil, phones, tones in zip(w_ceil_list, phone_list, tone_list):
#     print('phones:', phones.flatten().tolist())
#     print('tones:', tones.flatten().tolist())
#     print('w_ceil:', w_ceil.flatten().int().tolist())

print(f'w_ceil_list: {w_ceil_list}')

df = pd.DataFrame({
    'phones': [id_to_symbol_map.get(item, '') for sublist in phone_list for item in sublist.flatten().tolist()],
    'tones': [item for sublist in tone_list for item in sublist.flatten().tolist()],
    'w_ceil': [item for sublist in w_ceil_list for item in sublist.flatten().int().tolist()]
})


df.T

d:\miniconda\envs\melotts\lib\site-packages\librosa\util\files.py:10: UserWarning: pkg_resources is deprecated as an API. See https://setuptools.pypa.io/en/latest/pkg_resources.html. The pkg_resources package is slated for removal as early as 2025-11-30. Refrain from using this package or pin to Setuptools<81.
  from pkg_resources import resource_filename
d:\miniconda\envs\melotts\lib\site-packages\tqdm\auto.py:21: TqdmWarning: IProgress not found. Please update jupyter and ipywidgets. See https://ipywidgets.readthedocs.io/en/stable/user_install.html
  from .autonotebook import tqdm as notebook_tqdm
d:\miniconda\envs\melotts\lib\site-packages\huggingface_hub\file_download.py:949: FutureWarning: `resume_download` is deprecated and will be removed in version 1.0.0. Downloads always resume when possible. If you want to force a new download, use `force_download=True`.
  warnings.warn(
d:\miniconda\envs\melotts\lib\site-packages\google\api_core\_python_version_support.py:242: FutureWarning:

 > Text split to sentences.
落霞与孤鹜齐飞, 秋水共长天一色.
 > ===========================


  0%|          | 0/1 [00:00<?, ?it/s]Building prefix dict from the default dictionary ...
Loading model from cache C:\Users\13081\AppData\Local\Temp\jieba.cache
Loading model cost 0.495 seconds.
Prefix dict has been built successfully.
d:\miniconda\envs\melotts\lib\site-packages\huggingface_hub\file_download.py:949: FutureWarning: `resume_download` is deprecated and will be removed in version 1.0.0. Downloads always resume when possible. If you want to force a new download, use `force_download=True`.
  warnings.warn(
Some weights of the model checkpoint at bert-base-multilingual-uncased were not used when initializing BertForMaskedLM: ['cls.seq_relationship.weight', 'cls.seq_relationship.bias']
- This IS expected if you are initializing BertForMaskedLM from the checkpoint of a model trained on another task or with another architecture (e.g. initializing a BertForSequenceClassification model from a BertForPreTraining model).
- This IS NOT expected if you are initializing BertForMaskedLM

id_to_symbol_map: {0: '_', 1: 'AA', 2: 'E', 3: 'EE', 4: 'En', 5: 'N', 6: 'OO', 7: 'V', 8: 'a', 9: 'a:', 10: 'aa', 11: 'ae', 12: 'ah', 13: 'ai', 14: 'an', 15: 'ang', 16: 'ao', 17: 'aw', 18: 'ay', 19: 'b', 20: 'by', 21: 'c', 22: 'ch', 23: 'd', 24: 'dh', 25: 'dy', 26: 'e', 27: 'e:', 28: 'eh', 29: 'ei', 30: 'en', 31: 'eng', 32: 'er', 33: 'ey', 34: 'f', 35: 'g', 36: 'gy', 37: 'h', 38: 'hh', 39: 'hy', 40: 'i', 41: 'i0', 42: 'i:', 43: 'ia', 44: 'ian', 45: 'iang', 46: 'iao', 47: 'ie', 48: 'ih', 49: 'in', 50: 'ing', 51: 'iong', 52: 'ir', 53: 'iu', 54: 'iy', 55: 'j', 56: 'jh', 57: 'k', 58: 'ky', 59: 'l', 60: 'm', 61: 'my', 62: 'n', 63: 'ng', 64: 'ny', 65: 'o', 66: 'o:', 67: 'ong', 68: 'ou', 69: 'ow', 70: 'oy', 71: 'p', 72: 'py', 73: 'q', 74: 'r', 75: 'ry', 76: 's', 77: 'sh', 78: 't', 79: 'th', 80: 'ts', 81: 'ty', 82: 'u', 83: 'u:', 84: 'ua', 85: 'uai', 86: 'uan', 87: 'uang', 88: 'uh', 89: 'ui', 90: 'un', 91: 'uo', 92: 'uw', 93: 'v', 94: 'van', 95: 've', 96: 'vn', 97: 'w', 98: 'x', 99: 'y', 100: 

,0,1,2,3,4,5,6,7,8,9,10,11,12,13,14,15,16,17,18,19,20,21,22,23,24,25,26,27,28,29,30,31,32,33,34,35,36,37,38,39,40,41,42,43,44,45,46,47,48,49,50,51,52,53,54,55,56,57,58,59,60,61,62,63,64
phones,_,_,_,l,_,uo,_,x,_,ia,_,y,_,v,_,g,_,u,_,w,_,u,_,q,_,i,_,f,_,ei,_,",",_,q,_,iu,_,sh,_,ui,_,g,_,ong,_,ch,_,ang,_,t,_,ian,_,y,_,i,_,s,_,e,_,.,_,_,_
tones,0,0,0,4,0,4,0,2,0,2,0,3,0,3,0,1,0,1,0,4,0,4,0,2,0,2,0,1,0,1,0,0,0,1,0,1,0,3,0,3,0,4,0,4,0,2,0,2,0,1,0,1,0,2,0,2,0,4,0,4,0,0,0,0,0
w_ceil,7,7,15,2,3,3,2,6,4,5,3,4,4,3,6,3,3,3,3,3,3,3,3,11,4,3,3,4,6,6,4,4,7,5,2,5,3,8,4,4,4,3,4,4,3,4,4,6,4,5,3,3,3,3,4,3,5,6,5,5,4,3,3,3,4


从以上 pandas DataFrame 中可以看到每个音素对应的持续时间（w_ceil），w_ceil 的一个单位对应 512 (hop_length) / 44100 (采样率) ≈ 0.0116s = 11.6ms 的时间长度。因此，可以通过调整 `w_ceil_list` 中每个 w_ceil 的值，来控制每个音素的持续时间，从而实现简单的韵律控制。

In [2]:
# w_ceil_list = [[7, 7, 15, 2, 3, 3, 2, 6, 4, 5, 3, 4, 4, 3, 6, 3, 9, 9, 9, 9, 9, 9, 9, 11, 4, 3, 3, 4, 6, 6, 4, 4, 7, 5, 2, 5, 3, 8, 4, 4, 4, 3, 4, 4, 3, 8, 8, 18, 12, 15, 12, 12, 12, 3, 4, 3, 5, 6, 5, 5, 4, 3, 3, 3, 4]]
# w_ceil_list = [[7, 7, 15, 2, 3, 3, 2, 6, 4, 5, 3, 4, 4, 3, 6, 3, 9, 9, 9, 9, 9, 9, 9, 11, 4, 3, 3, 4, 6, 6, 4, 4, 7, 5, 2, 5, 3, 8, 4, 4, 4, 3, 4, 4, 3, 8, 8, 18, 12, 15, 12, 12, 12, 3, 4, 3, 5, 6, 5, 5, 4, 3, 3, 3, 4]]
modified_w_ceil_list = w_ceil_list[0].squeeze(0).int().tolist()
modified_w_ceil_list[0][15:23] = [9] * 8 # 延长 “孤” 和 “鹜” 音素的持续时间为 9 个单位（约 104ms）
modified_w_ceil_list[0][45:53] = [9] * 8 # 延长 “长” 和 “天” 音素的持续时间为 9 个单位（约 104ms）

# 修改后的 w_ceil_list
print(f'modified w_ceil_list: {modified_w_ceil_list}')


modified w_ceil_list: [[7, 7, 15, 2, 3, 3, 2, 6, 4, 5, 3, 4, 4, 3, 6, 9, 9, 9, 9, 9, 9, 9, 9, 11, 4, 3, 3, 4, 6, 6, 4, 4, 7, 5, 2, 5, 3, 8, 4, 4, 4, 3, 4, 4, 3, 9, 9, 9, 9, 9, 9, 9, 9, 3, 4, 3, 5, 6, 5, 5, 4, 3, 3, 3, 4]]


In [3]:
# 按照原始的韵律生成语音，关闭随机性以便对比
model.tts_to_file(text, speaker_ids['ZH'], output_path, speed=speed, sdp_ratio=0, noise_scale=0, noise_scale_w=0)

from IPython.display import Audio

Audio(output_path)

 > Text split to sentences.
落霞与孤鹜齐飞, 秋水共长天一色.
 > ===========================


100%|██████████| 1/1 [00:00<00:00,  1.09it/s]


In [4]:
# 按照调整后的韵律生成语音，使用自定义的 w_ceil_list 来控制每个音素的持续时间，关闭随机性以便对比
model.tts_to_file_custom_duration(text, speaker_ids['ZH'], output_path, speed=speed, sdp_ratio=0, noise_scale=0, noise_scale_w=0, w_ceil_customized=modified_w_ceil_list)

from IPython.display import Audio

Audio(output_path)

 > Text split to sentences.
落霞与孤鹜齐飞, 秋水共长天一色.
 > ===========================


100%|██████████| 1/1 [00:01<00:00,  1.12s/it]


可以观察到修改后的音频中，"孤"和"鹜"这两个字的音素的持续时间被延长了，"长"和"天"这两个字的音素的持续时间也被延长了，从而产生更突出的节奏和停连效果。

> 【问题三】 请通过调整 `w_ceil` 来控制合成语音的韵律，合成一句 “你听到了吗？这句话我会说得越来越慢。” 的语音。要求语速越来越慢：句首接近原始时长、句尾约为原始时长的 3 倍（对应语速约从 1 倍`线性`降至 0.33 倍）。提交代码和生成的音频文件（命名为 `homework_3.wav`）。

In [5]:
from melo.api import TTS
import numpy as np

# Speed is adjustable
speed = 1.0
device = 'cpu'  # or cuda:0
text = "你听到了吗？这句话我会说得越来越慢。"

model = TTS(language='ZH', device=device)
speaker_ids = model.hps.data.spk2id

output_path = 'homework_3.wav'

# 获取原始 w_ceil
w_ceil_list, phone_list, tone_list = model.get_original_w_ceil(
    text, speaker_ids['ZH'], output_path, speed=speed, sdp_ratio=0, noise_scale=0, noise_scale_w=0
 )

# 展开为一条序列，做线性变速（1.0 -> 3.0）
w_ceil_flat = [int(x) for sub in w_ceil_list for x in sub.flatten().int().tolist()]
n = len(w_ceil_flat)
scale = np.linspace(1.0, 3.0, n)
w_ceil_scaled = [max(1, int(round(w * s))) for w, s in zip(w_ceil_flat, scale)]

# 按原句拆分回列表
lengths = [len(sub.flatten().int().tolist()) for sub in w_ceil_list]
w_ceil_customized = []
idx = 0
for L in lengths:
    w_ceil_customized.append(w_ceil_scaled[idx:idx + L])
    idx += L

# 生成越来越慢的语音
model.tts_to_file_custom_duration(
    text,
    speaker_ids['ZH'],
    output_path,
    speed=speed,
    sdp_ratio=0,
    noise_scale=0,
    noise_scale_w=0,
    w_ceil_customized=w_ceil_customized,
 )

from IPython.display import Audio
Audio(output_path)

d:\miniconda\envs\melotts\lib\site-packages\torch\nn\utils\weight_norm.py:143: FutureWarning: `torch.nn.utils.weight_norm` is deprecated in favor of `torch.nn.utils.parametrizations.weight_norm`.
  WeightNorm.apply(module, name, dim)


 > Text split to sentences.
你听到了吗. 这句话我会说得越来越慢.
 > ===========================


100%|██████████| 1/1 [00:00<00:00,  7.34it/s]


 > Text split to sentences.
你听到了吗. 这句话我会说得越来越慢.
 > ===========================


100%|██████████| 1/1 [00:01<00:00,  1.67s/it]
